###Loading data.

In [1]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
path = kagglehub.competition_download('filament-segmentation-2026')

print("Path to competition files:", path)

100%|██████████| 671M/671M [00:03<00:00, 226MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/filament-segmentation-2026


In [4]:
def generate_mask_for_image(img_info, filaments):
    """
    It creates a binary mask (0 and 1).
    """
    height = img_info['height']
    width = img_info['width']

    # Background
    mask = np.zeros((height, width), dtype=np.uint8)

    for ann in filaments:
        segmentations = ann.get('segmentation', [])

        for seg in segmentations:
            poly = np.array(seg, dtype=np.int32).reshape((-1, 2))

            cv2.fillPoly(mask, [poly], color=1)

    return mask


In [5]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class FilamentDataGenerator(Sequence):
    def __init__(self, images_list, annotations_map, img_dir, batch_size=8, target_size=(512, 512), shuffle=True):
        self.images_list = images_list
        self.annotations_map = annotations_map
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.images_list))
        self.on_epoch_end()

    def __len__(self):
      """
      Number of batch per epoch.
      """
      return int(np.ceil(len(self.images_list) / self.batch_size))

    def on_epoch_end(self):
      """
      Randomly shuffle the data for each epoch.
      """
      if self.shuffle:
          np.random.shuffle(self.indexes)

    def __getitem__(self, index):
      """
      Reads and resizes 8 photos and smoothes them.
      """

      batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

      batch_images = []
      batch_masks = []

      for idx in batch_indexes:
            img_info = self.images_list[idx]
            img_id = img_info['id']

            # Reading an image from disk.
            img_path = os.path.join(self.img_dir, img_info['file_name'])
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Generating a binary mask of the same image.
            filaments = self.annotations_map.get(img_id, [])
            mask = generate_mask_for_image(img_info, filaments)

            # On-the-fly resizing.
            image_resized = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
            mask_resized = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

            # Data normalization (numbers between 0 and 1).
            image_normalized = image_resized.astype(np.float32)
            mask_normalized = np.expand_dims(mask_resized.astype(np.float32), axis=-1)

            batch_images.append(image_normalized)
            batch_masks.append(mask_normalized)

      return np.array(batch_images), np.array(batch_masks)

In [6]:
from sklearn.model_selection import train_test_split
import os
import json
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path
BASE_DIR = '/root/.cache/kagglehub/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
ANN_PATH = os.path.join(BASE_DIR, 'train', 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
# Reading json path
with open(ANN_PATH, 'r') as f:
    coco_data = json.load(f)
all_images = coco_data['images']

train_images, val_images = train_test_split(all_images, test_size=0.25, random_state=19)
IMG_DIR = os.path.join(BASE_DIR, 'train', 'train_images')

from collections import defaultdict

filaments_by_image = defaultdict(list)
for ann in coco_data['annotations']:
    filaments_by_image[ann['image_id']].append(ann)

train_generator = FilamentDataGenerator(
    images_list=train_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=8,
    target_size=(512, 512),
    shuffle=True
)

val_generator = FilamentDataGenerator(
    images_list=val_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=8,
    target_size=(512, 512),
    shuffle=False
)

print(f"Number of batch Train: {len(train_generator)}")
print(f"Number of batch Validation: {len(val_generator)}")

Number of batch Train: 109
Number of batch Validation: 37


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0

encoder = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(512, 512, 3)
)

encoder.trainable = False

skip_names = [
    "block2a_expand_activation",
    "block3a_expand_activation",
    "block4a_expand_activation",
    "block6a_expand_activation"
]

skip_outputs = [
    encoder.get_layer(name).output
    for name in skip_names
]

encoder_model = Model(
    inputs=encoder.input,
    outputs=skip_outputs
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
def decoder_block(x, skip, filters):

    x = layers.Conv2DTranspose(
        filters,
        (2, 2),
        strides=(2, 2),
        padding="same"
    )(x)

    x = layers.Concatenate()([x, skip])

    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    return x

In [9]:
def build_transfer_unet():

    inputs = layers.Input(shape=(512, 512, 3))

    skip1, skip2, skip3, skip4 = encoder_model(inputs)

    x = skip4

    x = decoder_block(x, skip3, 256)

    x = decoder_block(x, skip2, 128)

    x = decoder_block(x, skip1, 64)

    x = layers.Conv2DTranspose(
        32,
        (2, 2),
        strides=(2, 2),
        padding="same"
    )(x)

    x = layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    outputs = layers.Conv2D(
        1,
        (1, 1),
        activation="sigmoid"
    )(x)

    return Model(inputs, outputs)

In [10]:
import tensorflow.keras.backend as K

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = 1.0 - dice_coef(y_true, y_pred)
    return bce + dice

In [11]:
transfer_model = build_transfer_unet()

In [12]:
print(transfer_model.output_shape)

(None, 512, 512, 1)


In [13]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=bce_dice_loss,
    metrics=[dice_coef]
)

In [15]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    ReduceLROnPlateau(monitor='val_dice_coef', factor=0.5, patience=3, min_lr=1e-5, mode='max', verbose=1),
    ModelCheckpoint('best_filament_model.keras', monitor='val_dice_coef', save_best_only=True, mode='max', verbose=1)
]

history_transfer = transfer_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step - dice_coef: 0.0287 - loss: 1.2928
Epoch 1: val_dice_coef improved from None to 0.07692, saving model to best_filament_model.keras

Epoch 1: finished saving model to best_filament_model.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 161s 855ms/step - dice_coef: 0.1072 - loss: 1.0534 - val_dice_coef: 0.0769 - val_loss: 0.9459 - learning_rate: 1.0000e-04
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - dice_coef: 0.5540 - loss: 0.4577
Epoch 2: val_dice_coef improved from 0.07692 to 0.50742, saving model to best_filament_model.keras

Epoch 2: finished saving model to best_filament_model.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 63s 575ms/step - dice_coef: 0.5917 - loss: 0.4202 - val_dice_coef: 0.5074 - val_loss: 0.5036 - learning_rate: 1.0000e-04
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 466ms/step - dice_coef: 0.6421 - loss: 0.3718
Epoch 3: val_dice_coef did not improve from 0.50742
109/109 ━━━━━━━━━━━━━━━━━━━━ 63s 575ms/step - dice_coef: 0.6449

In [ ]:
print(history_transfer.history.keys())

dict_keys(['dice_coef', 'loss', 'val_dice_coef', 'val_loss'])


In [ ]:
import plotly.express as px
import plotly
import plotly.io as pio

epochs = list(range(1, len(history_transfer.history['loss']) + 1))

fig_loss = px.line(
    x=epochs,
    y=[history_transfer.history['loss'], history_transfer.history['val_loss']],
    labels={'x': 'Epoch', 'value': 'Loss', 'variable': 'Dataset'},
    title='Loss Curve',
    markers=True
)
fig_loss.show()

In [ ]:
fig_dice = px.line(
    x=epochs,
    y=[history_transfer.history['dice_coef'], history_transfer.history['val_dice_coef']],
    labels={'x': 'Epoch', 'value': 'Dice Score', 'variable': 'Dataset'},
    title='Dice Score',
    markers=True
)
fig_dice.show()

In [ ]:
import random
sample_val_images = random.sample(val_images, 100)

sample = FilamentDataGenerator(
  images_list=sample_val_images,
  annotations_map=filaments_by_image,
  img_dir=IMG_DIR,
  batch_size=10,
  target_size=(512, 512),
  shuffle=True
)

x_batch, y_true = sample[0]

In [ ]:
predictions = transfer_model.predict(x_batch)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step


In [ ]:
y_pred = (predictions > 0.45).astype(int)
y_true = (y_true > 0.45).astype(int)

In [ ]:
def dice_metric(y_true, y_pred, smooth=2e-4):
  """
  Dice = (2 x |Y ∩ Ŷ|) ÷ (|Y| + |Ŷ|) = (2 x TP) ÷ (2 x TP + FP + FN)
  """
  intersection = np.sum(y_true * y_pred)  # |Y ∩ Ŷ|
  total = np.sum(y_true) + np.sum(y_pred)  # |Y| + |Ŷ|

  return (2.0 * intersection + smooth) / (total + smooth)

In [ ]:
print(f"Dice metric result: {(dice_metric(y_true, y_pred) * 100):.2f}")

Dice metric result: 66.48
